In [ ]:
!pip install -q kaggle tensorflow matplotlib  seaborn

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
!unzip intel-image-classification.zip -d /content/my_extracted_folder


In [ ]:
!find . -maxdepth 3 -type d | head -50

In [ ]:
train_dir =r'/content/my_extracted_folder/seg_train/seg_train'
test_dir =r'/content/my_extracted_folder/seg_test/seg_test'

In [ ]:
classes = sorted(os.listdir(train_dir))

print(classes)
print("Number of classes:", len(classes))

In [ ]:
# Image count
for class_name in classes :
  class_path = os.path.join(train_dir,class_name )
  count= len(os.listdir(class_path))
  print(class_name ,":", count)

In [ ]:
plt.figure(figsize=(15, 8))

for i, class_name in enumerate(classes):

    class_path = os.path.join(train_dir, class_name)

    image_name = os.listdir(class_path)[0]

    image_path = os.path.join(class_path, image_name)

    image = plt.imread(image_path)

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

# Image Preprocessing

      224X224 is a standard input size for imagenet-pretrained vision  model and is directly supported by mobileNetv2

In [ ]:
img_size=(224,224)
batch_size=32

In [ ]:
# Data Generator
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
train_datagen

In [ ]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

In [ ]:
# validation data
validation_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

In [ ]:
print(train_data.class_indices)

In [ ]:
# load pretraind model --. MobileNetV2
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(6, activation="softmax")(x)


In [ ]:
model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
history= model.fit(train_data,validation_data=validation_data,epochs=2)

In [ ]:
# Plot Accuracy
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# plot loss
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Test Dataset
test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
# evaluate the model
test_loss, test_accuracy = model.evaluate(test_data)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
# confusion Matrix
predictions = model.predict(test_data)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_data.classes

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")

In [ ]:
# classification report
from sklearn.metrics import classification_report
class_report = classification_report(true_classes, predicted_classes, target_names=classes)
print("Classification Report:\n", class_report)

In [ ]:
# predict a single image
from tensorflow.keras.utils import load_img, img_to_array
image_path= "/content/my_extracted_folder/seg_test/seg_test/sea/20072.jpg"
img = load_img(image_path, target_size=img_size)
img_array=img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
prediction = model.predict(img_array)

In [ ]:
predicted_class_index = np.argmax(prediction)
predicted_class = list(train_data.class_indices.keys())[predicted_class_index]
print("Predicted Class:", predicted_class)

In [ ]:
# display prediction
plt.imshow(img)
plt.axis("off")
plt.title(f"Predicted Class: {predicted_class}")
plt.axis('off')
plt.show()

In [ ]:
!pip install -q gradio

In [ ]:
import tensorflow as tf
import numpy as np
import gradio as gr

# The model object is already available in the kernel from previous cells
# model = tf.keras.models.load_model(
#     "/content/intel_mobilenetv2_model.keras"
# )

classes = [
    "buildings",
    "forest",
    "glacier",
    "mountain",
    "sea",
    "street"
]

IMG_SIZE = (224, 224)

In [ ]:
def predict_image(image):

    # Resize image
    image = image.resize(IMG_SIZE)

    # Convert image to NumPy array
    image = np.array(image)

    # Add batch dimension
    image = np.expand_dims(image, axis=0)

    # MobileNetV2 preprocessing
    image = tf.keras.applications.mobilenet_v2.preprocess_input(
        image
    )

    # Prediction
    prediction = model.predict(image, verbose=0)[0]

    # Get class
    predicted_index = np.argmax(prediction)

    predicted_class = classes[predicted_index]

    confidence = float(prediction[predicted_index])

    return {
        classes[i]: float(prediction[i])
        for i in range(len(classes))
    }

In [ ]:
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(
        type="pil",
        label="Upload Scene Image"
    ),
    outputs=gr.Label(
        num_top_classes=6,
        label="Prediction"
    ),
    title="Intel Scene Classification",
    description="Upload an image to classify it into one of six scene categories using MobileNetV2 Transfer Learning."
)

In [ ]:
demo.launch(share=True)